# LeetCode 3712 — Sum of Elements With Frequency Divisible by K bằng Python

Link đề: https://leetcode.com/problems/sum-of-elements-with-frequency-divisible-by-k/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
class Solution:
    def sumDivisibleByK(self, nums: List[int], k: int) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một giá trị `v` được tính nếu **số lần xuất hiện** (tần suất) của nó chia hết cho `k`; khi được tính, `v` đóng góp **mỗi lần xuất hiện** vào tổng (tức `v * tần_suất`).
2. Biết đếm tần suất bằng `dict`, `collections.Counter`, mảng đếm và `sorted` + `itertools.groupby`.
3. Phân biệt `freq % k == 0` ("tần suất chia hết cho k") với `k % freq == 0` (chiều ngược lại), và phân biệt "cộng mỗi giá trị một lần" với "cộng theo số lần xuất hiện".
4. Viết 5 cách khác nhau (đếm bằng `count`, `Counter`, `dict` thủ công, `groupby`, mảng đếm 101 ô) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: quên nhân tần suất, đảo chiều phép chia hết, bỏ sót phần tử, và sửa dict khi đang duyệt.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và một số nguyên `k`. **Tần suất** của một giá trị là số lần giá trị đó xuất hiện trong `nums`. Hãy trả về tổng của tất cả các phần tử mà **tần suất của giá trị đó chia hết cho `k`**. Mỗi lần xuất hiện của giá trị đủ điều kiện đều được cộng. Nếu không có giá trị nào đủ điều kiện, trả về `0`.

| Ví dụ | nums, k | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,2,2,3,3,3,3,4]`, `2` | `16` | `2` xuất hiện 2 lần, `3` xuất hiện 4 lần (đều chẵn): `2+2+3+3+3+3 = 16` |
| 2 | `[1,2,3,4,5]`, `2` | `0` | Mỗi giá trị xuất hiện 1 lần, không chia hết cho 2 |
| 3 | `[4,4,4,1,2,3]`, `3` | `12` | `4` xuất hiện 3 lần: `4+4+4 = 12` |

Constraints: `1 <= nums.length <= 100`, `1 <= nums[i] <= 100`, `1 <= k <= 100`.

In [1]:
from collections import Counter

for nums, k in [([1, 2, 2, 3, 3, 3, 3, 4], 2), ([1, 2, 3, 4, 5], 2), ([4, 4, 4, 1, 2, 3], 3)]:
    cnt = Counter(nums)
    picked = {v: c for v, c in cnt.items() if c % k == 0}
    print(f"nums={nums} k={k}  tần suất={dict(cnt)}  đủ điều kiện={picked}  tổng={sum(v * c for v, c in picked.items())}")

nums=[1, 2, 2, 3, 3, 3, 3, 4] k=2  tần suất={1: 1, 2: 2, 3: 4, 4: 1}  đủ điều kiện={2: 2, 3: 4}  tổng=16
nums=[1, 2, 3, 4, 5] k=2  tần suất={1: 1, 2: 1, 3: 1, 4: 1, 5: 1}  đủ điều kiện={}  tổng=0
nums=[4, 4, 4, 1, 2, 3] k=3  tần suất={4: 3, 1: 1, 2: 1, 3: 1}  đủ điều kiện={4: 3}  tổng=12


## 3. Kiến thức nền

### 3.1. Tần suất và `list.count`

`lst.count(x)` trả về số lần `x` xuất hiện trong `lst` (quét cả list, `O(n)`). Dùng được cho bài nhỏ, nhưng nếu gọi cho mọi phần tử sẽ lặp lại công việc.

In [2]:
nums = [1, 2, 2, 3, 3, 3, 3, 4]
print(nums.count(2), nums.count(3), nums.count(9))
print([nums.count(x) for x in nums])

2 4 0
[1, 2, 2, 4, 4, 4, 4, 1]


### 3.2. `dict` để đếm

`dict` ánh xạ **khóa -> giá trị**. Để đếm: với mỗi `v`, tăng `cnt[v]`. Phương thức `dict.get(key, default)` trả `default` khi khóa chưa có, tránh lỗi `KeyError`.

In [3]:
nums = [4, 4, 4, 1, 2, 3]
cnt = {}
for v in nums:
    cnt[v] = cnt.get(v, 0) + 1
print(cnt)
print(cnt.get(4, 0), cnt.get(99, 0))
print(cnt.items())

{4: 3, 1: 1, 2: 1, 3: 1}
3 0
dict_items([(4, 3), (1, 1), (2, 1), (3, 1)])


### 3.3. `collections.Counter`

`Counter(iterable)` là `dict` chuyên đếm: `Counter(nums)[v]` là tần suất của `v` (bằng `0` nếu không có, không báo lỗi). `.items()` cho cặp `(giá trị, tần suất)`; `.values()` cho các tần suất.

In [4]:
from collections import Counter
c = Counter([4, 4, 4, 1, 2, 3])
print(c)
print(c[4], c[100])
print(list(c.items()), list(c.values()))
print(c.most_common(1))

Counter({4: 3, 1: 1, 2: 1, 3: 1})
3 0
[(4, 3), (1, 1), (2, 1), (3, 1)] [3, 1, 1, 1]
[(4, 3)]


### 3.4. Phép chia hết với `%`

`a % k == 0` nghĩa là `a` chia hết cho `k`. Chú ý **chiều**: "tần suất chia hết cho `k`" là `freq % k == 0`, khác với `k % freq == 0` ("`k` chia hết cho tần suất").

In [5]:
freq, k = 4, 2
print(freq % k == 0, k % freq == 0)     # True False: hai chiều khác nhau
freq, k = 1, 3
print(freq % k == 0, k % freq == 0)     # False True
print([f for f in range(1, 13) if f % 3 == 0])

True False
False True
[3, 6, 9, 12]


### 3.5. `itertools.groupby` trên danh sách đã sắp xếp

Sau khi `sorted(nums)`, các giá trị bằng nhau nằm **liền nhau**. `groupby(sorted_nums)` gom chúng thành từng nhóm `(giá trị, iterator các phần tử)`. Độ dài nhóm chính là tần suất. Lưu ý: `groupby` chỉ gom các phần tử **liền kề**, nên phải sắp xếp trước.

In [6]:
from itertools import groupby
a = sorted([4, 4, 4, 1, 2, 3])
for key, grp in groupby(a):
    items = list(grp)
    print(key, items, "tần suất =", len(items))
print([(k, len(list(g))) for k, g in groupby([1, 2, 1])])   # chưa sắp xếp: 1 bị tách làm hai nhóm

1 [1] tần suất = 1
2 [2] tần suất = 1
3 [3] tần suất = 1
4 [4, 4, 4] tần suất = 3
[(1, 1), (2, 1), (1, 1)]


### 3.6. Mảng đếm (counting array)

Vì `1 <= nums[i] <= 100`, ta có thể dùng list `cnt = [0] * 101` và `cnt[v] += 1`. Chỉ số chính là giá trị, nên không cần băm.

In [7]:
cnt = [0] * 101
for v in [4, 4, 4, 1, 2, 3]:
    cnt[v] += 1
print([(v, c) for v, c in enumerate(cnt) if c])

[(1, 1), (2, 1), (3, 1), (4, 3)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy hình dung lớp học có các bạn mặc áo ghi số `v`. Bạn đếm xem mỗi số áo có bao nhiêu bạn. Nhóm nào có **số bạn chia hết cho `k`** thì tất cả bạn trong nhóm đó "đóng góp" số áo của mình vào tổng. Vậy mỗi nhóm đóng góp `v * c` (với `c` là số bạn trong nhóm) nếu `c % k == 0`.

Tính tay ví dụ 1 (`k = 2`): `1 -> 1 lần` (lẻ, bỏ), `2 -> 2 lần` (chẵn, `2*2 = 4`), `3 -> 4 lần` (chẵn, `3*4 = 12`), `4 -> 1 lần` (bỏ). Tổng `4 + 12 = 16`.

Hai bước: (1) **đếm** tần suất; (2) **cộng** `v * c` cho nhóm đạt `c % k == 0`.

### Bảng tính tay

| nums, k | tần suất | nhóm đạt | tổng |
|---|---|---|---|
| `[1,2,2,3,3,3,3,4], 2` | 1:1, 2:2, 3:4, 4:1 | 2 (x2), 3 (x4) | 4 + 12 = 16 |
| `[1,2,3,4,5], 2` | mỗi giá trị 1 | không | 0 |
| `[4,4,4,1,2,3], 3` | 4:3, 1:1, 2:1, 3:1 | 4 (x3) | 12 |
| `[5,5], 1` | 5:2 | 5 (x2), vì mọi số chia hết cho 1 | 10 |

In [8]:
from collections import Counter
for nums, k in [([1, 2, 2, 3, 3, 3, 3, 4], 2), ([1, 2, 3, 4, 5], 2), ([4, 4, 4, 1, 2, 3], 3), ([5, 5], 1)]:
    cnt = Counter(nums)
    total = sum(v * c for v, c in cnt.items() if c % k == 0)
    print(nums, k, dict(cnt), "->", total)

[1, 2, 2, 3, 3, 3, 3, 4] 2 {1: 1, 2: 2, 3: 4, 4: 1} -> 16
[1, 2, 3, 4, 5] 2 {1: 1, 2: 1, 3: 1, 4: 1, 5: 1} -> 0
[4, 4, 4, 1, 2, 3] 3 {4: 3, 1: 1, 2: 1, 3: 1} -> 12
[5, 5] 1 {5: 2} -> 10


## 5. Cách 1 — Brute Force / baseline

Với **mỗi phần tử** `x` trong `nums`, tính `nums.count(x)`; nếu chia hết cho `k` thì cộng `x` vào tổng. Mỗi lần xuất hiện của `x` đều được duyệt riêng nên tự động cộng đủ `x * tần_suất` lần.

Nút thắt: `count` quét cả mảng cho **mỗi** phần tử nên tổng `O(n²)`; cùng một giá trị bị đếm lặp lại nhiều lần.

In [9]:
class SolutionBrute:
    def sumDivisibleByK(self, nums, k):
        total = 0
        for x in nums:
            if nums.count(x) % k == 0:      # đếm lại từ đầu cho mỗi phần tử
                total += x
        return total

print(SolutionBrute().sumDivisibleByK([1, 2, 2, 3, 3, 3, 3, 4], 2),
      SolutionBrute().sumDivisibleByK([1, 2, 3, 4, 5], 2),
      SolutionBrute().sumDivisibleByK([4, 4, 4, 1, 2, 3], 3))

16 0 12


**Độ phức tạp:** Time `O(n²)`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** đếm tần suất **một lần** cho mọi giá trị (`O(n)`), rồi duyệt các **giá trị khác nhau** thay vì từng phần tử.

**Quy tắc "nếu… thì…":**

- Nếu `freq[v] % k == 0` -> cộng `v * freq[v]` (không phải chỉ `v`).
- Nếu `k = 1` -> mọi giá trị đều đạt, đáp án là `sum(nums)`.
- Nếu `k > n` -> không giá trị nào có tần suất `>= k`, mà tần suất `>= 1` nên `freq % k = freq != 0`, đáp án `0`.
- Giá trị có tần suất `0` không xuất hiện, nên không thể vô tình được tính (dù `0 % k == 0`).

**Invariant:** sau khi duyệt xong một nhóm giá trị, `total` bằng tổng đóng góp của các giá trị đã xét.

In [10]:
# Kiểm chứng nhanh: nếu k = 1 thì đáp án = sum(nums); nếu k > len(nums) thì đáp án = 0
import random
from collections import Counter
random.seed(3712)
def ref(nums, k):
    return sum(v * c for v, c in Counter(nums).items() if c % k == 0)
for _ in range(1000):
    n = random.randint(1, 100)
    nums = [random.randint(1, 100) for _ in range(n)]
    assert ref(nums, 1) == sum(nums)
    if n < 100:
        assert ref(nums, n + 1) == 0
print("k=1 -> sum(nums); k>n -> 0: đúng")

k=1 -> sum(nums); k>n -> 0: đúng


## 7. Các cách tiếp theo

### Cách 2 — `Counter`

`Counter(nums)` đếm một lần; sau đó cộng `v * c` cho mọi cặp đạt điều kiện. Đây là cách thông dụng nhất.

In [11]:
from collections import Counter

class SolutionCounter:
    def sumDivisibleByK(self, nums, k):
        return sum(v * c for v, c in Counter(nums).items() if c % k == 0)

print(SolutionCounter().sumDivisibleByK([1, 2, 2, 3, 3, 3, 3, 4], 2),
      SolutionCounter().sumDivisibleByK([1, 2, 3, 4, 5], 2),
      SolutionCounter().sumDivisibleByK([4, 4, 4, 1, 2, 3], 3))

16 0 12


**Độ phức tạp:** Time `O(n)`, Space `O(d)` với `d` là số giá trị khác nhau (`d <= n`).

### Cách 3 — `dict` tự đếm

Giống `Counter` nhưng tự viết bằng `dict.get`. Giúp hiểu cơ chế bên dưới và hữu ích khi không được dùng thư viện.

In [12]:
class SolutionDict:
    def sumDivisibleByK(self, nums, k):
        freq = {}
        for v in nums:
            freq[v] = freq.get(v, 0) + 1
        total = 0
        for v, c in freq.items():
            if c % k == 0:
                total += v * c
        return total

print(SolutionDict().sumDivisibleByK([1, 2, 2, 3, 3, 3, 3, 4], 2),
      SolutionDict().sumDivisibleByK([1, 2, 3, 4, 5], 2),
      SolutionDict().sumDivisibleByK([4, 4, 4, 1, 2, 3], 3))

16 0 12


**Độ phức tạp:** Time `O(n)`, Space `O(d)`.

### Cách 4 — Sắp xếp + `groupby`

Sắp xếp để các giá trị bằng nhau nằm liền kề, rồi `groupby` đo độ dài mỗi nhóm. Không cần `dict`, nhưng tốn `O(n log n)` cho việc sắp xếp.

In [13]:
from itertools import groupby

class SolutionGroupby:
    def sumDivisibleByK(self, nums, k):
        total = 0
        for v, grp in groupby(sorted(nums)):
            c = len(list(grp))
            if c % k == 0:
                total += v * c
        return total

print(SolutionGroupby().sumDivisibleByK([1, 2, 2, 3, 3, 3, 3, 4], 2),
      SolutionGroupby().sumDivisibleByK([1, 2, 3, 4, 5], 2),
      SolutionGroupby().sumDivisibleByK([4, 4, 4, 1, 2, 3], 3))

16 0 12


**Độ phức tạp:** Time `O(n log n)`, Space `O(n)` (bản sắp xếp).

### Cách 5 — Mảng đếm 101 ô

Vì `nums[i] <= 100`, dùng list cố định kích thước `101`. Nhanh, không băm, bộ nhớ hằng số.

In [14]:
class SolutionArray:
    def sumDivisibleByK(self, nums, k):
        cnt = [0] * 101
        for v in nums:
            cnt[v] += 1
        return sum(v * c for v, c in enumerate(cnt) if c and c % k == 0)

print(SolutionArray().sumDivisibleByK([1, 2, 2, 3, 3, 3, 3, 4], 2),
      SolutionArray().sumDivisibleByK([1, 2, 3, 4, 5], 2),
      SolutionArray().sumDivisibleByK([4, 4, 4, 1, 2, 3], 3))

16 0 12


**Độ phức tạp:** Time `O(n + 101)`, Space `O(1)`. Chú ý điều kiện `c and ...`: ô có `c = 0` cũng thỏa `0 % k == 0` nhưng đóng góp `v * 0 = 0` nên vô hại; vẫn nên loại rõ ràng cho dễ hiểu.

## 8. TRACE từng bước

Trace Cách 3 (`dict`) với `nums = [4, 4, 4, 1, 2, 3]`, `k = 3`:

| bước | v | `freq` sau bước |
|---|---|---|
| 1 | 4 | `{4: 1}` |
| 2 | 4 | `{4: 2}` |
| 3 | 4 | `{4: 3}` |
| 4 | 1 | `{4: 3, 1: 1}` |
| 5 | 2 | `{4: 3, 1: 1, 2: 1}` |
| 6 | 3 | `{4: 3, 1: 1, 2: 1, 3: 1}` |

Giai đoạn cộng: `4` có `3 % 3 == 0` nên cộng `4 * 3 = 12`; `1, 2, 3` có `1 % 3 = 1` nên bỏ. Tổng `12`.

In [15]:
def trace_dict(nums, k):
    freq = {}
    print("nums =", nums, " k =", k)
    for step, v in enumerate(nums, 1):
        freq[v] = freq.get(v, 0) + 1
        print(f"  đếm bước {step}: v={v} -> freq={freq}")
    total = 0
    for v, c in freq.items():
        ok = (c % k == 0)
        if ok:
            total += v * c
        print(f"  giá trị {v}: tần suất {c}, {c} % {k} = {c % k} -> {'CỘNG ' + str(v * c) if ok else 'bỏ'}  total={total}")
    return total

assert trace_dict([4, 4, 4, 1, 2, 3], 3) == 12
assert trace_dict([1, 2, 2, 3, 3, 3, 3, 4], 2) == 16

nums = [4, 4, 4, 1, 2, 3]  k = 3
  đếm bước 1: v=4 -> freq={4: 1}
  đếm bước 2: v=4 -> freq={4: 2}
  đếm bước 3: v=4 -> freq={4: 3}
  đếm bước 4: v=1 -> freq={4: 3, 1: 1}
  đếm bước 5: v=2 -> freq={4: 3, 1: 1, 2: 1}
  đếm bước 6: v=3 -> freq={4: 3, 1: 1, 2: 1, 3: 1}
  giá trị 4: tần suất 3, 3 % 3 = 0 -> CỘNG 12  total=12
  giá trị 1: tần suất 1, 1 % 3 = 1 -> bỏ  total=12
  giá trị 2: tần suất 1, 1 % 3 = 1 -> bỏ  total=12
  giá trị 3: tần suất 1, 1 % 3 = 1 -> bỏ  total=12
nums = [1, 2, 2, 3, 3, 3, 3, 4]  k = 2
  đếm bước 1: v=1 -> freq={1: 1}
  đếm bước 2: v=2 -> freq={1: 1, 2: 1}
  đếm bước 3: v=2 -> freq={1: 1, 2: 2}
  đếm bước 4: v=3 -> freq={1: 1, 2: 2, 3: 1}
  đếm bước 5: v=3 -> freq={1: 1, 2: 2, 3: 2}
  đếm bước 6: v=3 -> freq={1: 1, 2: 2, 3: 3}
  đếm bước 7: v=3 -> freq={1: 1, 2: 2, 3: 4}
  đếm bước 8: v=4 -> freq={1: 1, 2: 2, 3: 4, 4: 1}
  giá trị 1: tần suất 1, 1 % 2 = 1 -> bỏ  total=0
  giá trị 2: tần suất 2, 2 % 2 = 0 -> CỘNG 4  total=4
  giá trị 3: tần suất 4, 4 % 2 = 0 -> C

In [16]:
def trace_groupby(nums, k):
    from itertools import groupby
    a = sorted(nums)
    print("sau sắp xếp:", a)
    total = 0
    for v, grp in groupby(a):
        c = len(list(grp))
        add = v * c if c % k == 0 else 0
        total += add
        print(f"  nhóm {v}: độ dài {c}, c % k = {c % k} -> +{add}  total={total}")
    return total

assert trace_groupby([4, 4, 4, 1, 2, 3], 3) == 12

sau sắp xếp: [1, 2, 3, 4, 4, 4]
  nhóm 1: độ dài 1, c % k = 1 -> +0  total=0
  nhóm 2: độ dài 1, c % k = 1 -> +0  total=0
  nhóm 3: độ dài 1, c % k = 1 -> +0  total=0
  nhóm 4: độ dài 3, c % k = 0 -> +12  total=12


## 9. Code nộp LeetCode

In [17]:
from typing import List, Optional
from collections import Counter

class Solution:
    def sumDivisibleByK(self, nums: List[int], k: int) -> int:
        total = 0
        for v, c in Counter(nums).items():
            if c % k == 0:
                total += v * c
        return total

## 10. Vì sao thuật toán đúng

- **Đếm đúng:** `Counter(nums)` ghi đúng số lần mỗi giá trị xuất hiện; tổng các tần suất bằng `n`.
- **Mỗi giá trị xét đúng một lần:** các giá trị khác nhau chia `nums` thành các nhóm rời nhau, nên không phần tử nào bị tính hai lần hay bỏ sót.
- **Đóng góp đúng:** theo đề, nếu tần suất của giá trị `v` chia hết cho `k` thì **mọi** lần xuất hiện của `v` đều được cộng, tức `c` lần `v`, bằng `v * c`. Ngược lại giá trị đó đóng góp `0`.
- **Không đủ điều kiện:** nếu không nhóm nào đạt thì tổng rỗng bằng `0`, đúng với yêu cầu "trả về 0".
- **Cách brute force:** với mỗi phần tử `x` cộng `x` khi `count(x) % k == 0`, tức mỗi lần xuất hiện của `x` được cộng một lần, cho cùng tổng `v * c`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| `count` cho từng phần tử | `O(n²)` | `O(1)` |
| `Counter` | `O(n)` | `O(d)` |
| `dict` thủ công | `O(n)` | `O(d)` |
| Sắp xếp + `groupby` | `O(n log n)` | `O(n)` |
| Mảng đếm 101 ô | `O(n + 101)` | `O(1)` |

`n = len(nums)`, `d` là số giá trị khác nhau. Với `n <= 100` mọi cách đều tức thì, nhưng lời giải thông thường là `O(n)` với bảng băm; mảng đếm chỉ dùng được vì giá trị bị chặn trên bởi `100`.

## 12. Bộ test

Ví dụ của đề, các biên (một phần tử, `k = 1`, `k` lớn hơn mọi tần suất, mọi phần tử bằng nhau, `n = 100`) và so sánh **tất cả cách** trên input ngẫu nhiên hợp lệ.

In [18]:
impls = [SolutionBrute(), SolutionCounter(), SolutionDict(), SolutionGroupby(), SolutionArray(), Solution()]

cases = [
    (([1, 2, 2, 3, 3, 3, 3, 4], 2), 16), (([1, 2, 3, 4, 5], 2), 0), (([4, 4, 4, 1, 2, 3], 3), 12),
    (([7], 1), 7), (([7], 2), 0), (([5, 5], 1), 10), (([5, 5], 2), 10), (([5, 5], 3), 0),
    (([100] * 100, 100), 10000), (([100] * 100, 99), 0), (([1] * 6 + [2] * 4, 3), 6), (([1] * 6 + [2] * 4, 2), 14),
    (([3, 3, 3, 3, 3, 3], 6), 18), (([1, 2, 3], 100), 0),
]
for (nums, k), exp in cases:
    assert 1 <= len(nums) <= 100 and all(1 <= v <= 100 for v in nums) and 1 <= k <= 100
    for impl in impls:
        got = impl.sumDivisibleByK(list(nums), k)
        assert got == exp, (type(impl).__name__, nums, k, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 14 ca


In [19]:
import random
random.seed(37120)
for _ in range(1000):
    n = random.randint(1, 100)
    hi = random.choice([3, 10, 100])                    # ít giá trị khác nhau -> tần suất lớn
    nums = [random.randint(1, hi) for _ in range(n)]
    k = random.randint(1, random.choice([3, 10, 100]))
    exp = sum(v * nums.count(v) for v in set(nums) if nums.count(v) % k == 0)
    outs = [impl.sumDivisibleByK(list(nums), k) for impl in impls]
    assert all(o == exp for o in outs), (nums, k, outs, exp)
    assert 0 <= outs[0] <= sum(nums)
print("1000 test ngẫu nhiên (seed=37120): 6 cách giống nhau")

1000 test ngẫu nhiên (seed=37120): 6 cách giống nhau


In [20]:
# Tính chất: kết quả không đổi khi xáo trộn thứ tự; k=1 cho sum(nums)
for _ in range(300):
    nums = [random.randint(1, 5) for _ in range(random.randint(1, 100))]
    k = random.randint(1, 6)
    a = Solution().sumDivisibleByK(nums, k)
    random.shuffle(nums)
    assert a == Solution().sumDivisibleByK(nums, k)
    assert Solution().sumDivisibleByK(nums, 1) == sum(nums)
print("bất biến theo thứ tự và trường hợp k=1 OK")

bất biến theo thứ tự và trường hợp k=1 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ cộng mỗi giá trị **một lần**

Đề yêu cầu cộng **mọi lần xuất hiện** (`v * c`), không phải chỉ `v`.

In [21]:
from collections import Counter
def wrong_once(nums, k):
    return sum(v for v, c in Counter(nums).items() if c % k == 0)     # SAI: thiếu * c

print(wrong_once([1, 2, 2, 3, 3, 3, 3, 4], 2), "!=", Solution().sumDivisibleByK([1, 2, 2, 3, 3, 3, 3, 4], 2))
assert wrong_once([1, 2, 2, 3, 3, 3, 3, 4], 2) == 5 != 16

5 != 16


### Lỗi 2: đảo chiều phép chia hết (`k % freq`)

"Tần suất chia hết cho k" là `freq % k == 0`. Viết ngược `k % freq == 0` sẽ chọn nhầm nhóm (ví dụ tần suất `1` luôn đạt vì `k % 1 == 0`).

In [22]:
def wrong_reversed_mod(nums, k):
    return sum(v * c for v, c in Counter(nums).items() if k % c == 0)   # SAI: đảo chiều

print(wrong_reversed_mod([4, 4, 4, 1, 2, 3], 3), "!=", Solution().sumDivisibleByK([4, 4, 4, 1, 2, 3], 3))
assert wrong_reversed_mod([4, 4, 4, 1, 2, 3], 3) == 18 != 12

18 != 12


### Lỗi 3: dùng `>= k` thay vì chia hết

"Chia hết cho `k`" khác với "ít nhất `k` lần". Tần suất `3` với `k = 2` không chia hết nhưng `3 >= 2`.

In [23]:
def wrong_at_least(nums, k):
    return sum(v * c for v, c in Counter(nums).items() if c >= k)       # SAI

print(wrong_at_least([7, 7, 7], 2), "!=", Solution().sumDivisibleByK([7, 7, 7], 2))
assert wrong_at_least([7, 7, 7], 2) == 21 != 0 == Solution().sumDivisibleByK([7, 7, 7], 2)

21 != 0


### Lỗi 4: dùng `groupby` mà chưa sắp xếp

`groupby` chỉ gom các phần tử **liền kề** bằng nhau; với mảng chưa sắp xếp, một giá trị bị tách thành nhiều nhóm và tần suất bị tính sai.

In [24]:
from itertools import groupby
def wrong_groupby_unsorted(nums, k):
    total = 0
    for v, grp in groupby(nums):                    # SAI: thiếu sorted
        c = len(list(grp))
        if c % k == 0:
            total += v * c
    return total

print(wrong_groupby_unsorted([2, 1, 2], 2), "!=", Solution().sumDivisibleByK([2, 1, 2], 2))
assert wrong_groupby_unsorted([2, 1, 2], 2) == 0 != 4

0 != 4


### Lỗi 5: sửa `dict` khi đang duyệt nó

Thêm/xóa khóa trong khi lặp `for v in freq` gây `RuntimeError: dictionary changed size during iteration`. Hãy duyệt trên bản sao hoặc chỉ đọc.

In [25]:
freq = {1: 1, 2: 2}
try:
    for v in freq:
        if freq[v] % 2 == 1:
            del freq[v]                              # SAI: sửa kích thước dict khi lặp
except RuntimeError as e:
    print("RuntimeError:", e)
good = {v: c for v, c in {1: 1, 2: 2}.items() if c % 2 == 0}    # cách đúng: tạo dict mới
print(good)
assert good == {2: 2}

RuntimeError: dictionary changed size during iteration
{2: 2}


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "tần suất / số lần xuất hiện" -> `Counter` ngay.
- Thấy "tổng các phần tử thỏa điều kiện về tần suất" -> cộng `v * c`, không phải `v`.
- Viết điều kiện bằng lời trước: "tần suất chia hết cho k" -> `c % k == 0` (tần suất đứng bên trái `%`).
- Biên nhanh: `k = 1` -> `sum(nums)`; `k` lớn hơn mọi tần suất -> `0`.
- Giá trị bị chặn nhỏ (<= 100) cho phép dùng mảng đếm thay `dict`.
- Có thể rút gọn một dòng: `sum(v * c for v, c in Counter(nums).items() if c % k == 0)`.

## 15. Biến thể của bài

1. **Sum of Unique Elements** (LeetCode 1748): cộng các giá trị xuất hiện đúng một lần.
2. **Sum of All Unique/Repeated Values**: cộng giá trị xuất hiện hơn một lần.
3. **Find Lucky Integer in an Array** (1394): số lớn nhất có tần suất bằng chính giá trị.
4. **Maximum Frequency Elements** (3005): tổng tần suất của các phần tử có tần suất lớn nhất.
5. **Majority Element** (169): phần tử xuất hiện nhiều hơn `n/2` lần.
6. **Top K Frequent Elements** (347): `k` phần tử có tần suất cao nhất (heap hoặc bucket sort).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| `count` từng phần tử | đếm lại cho mỗi phần tử | `O(n²)` | `O(1)` | Oracle kiểm tra, mảng rất nhỏ |
| `Counter` | đếm một lần rồi cộng `v * c` | `O(n)` | `O(d)` | Nộp bài, gọn và chuẩn |
| `dict` thủ công | `freq[v] = freq.get(v, 0) + 1` | `O(n)` | `O(d)` | Hiểu cơ chế, không dùng thư viện |
| Sắp xếp + `groupby` | nhóm giá trị liền kề | `O(n log n)` | `O(n)` | Khi muốn tránh bảng băm |
| Mảng đếm | `cnt[v] += 1` với `v <= 100` | `O(n + 101)` | `O(1)` | Giá trị chặn nhỏ, cần tốc độ |

Nộp cách `Counter`.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().sumDivisibleByK([2, 2, 2, 2, 5, 5], 2)` ra gì?

**Bài 2 (tính tay):** với `nums = [1, 1, 2, 3, 3, 3], k = 3`, liệt kê tần suất từng giá trị và tính tổng.

**Bài 3 (điền TODO):** hoàn thành `sum_unique(nums)` (LeetCode 1748): cộng các giá trị xuất hiện **đúng một lần**.

**Bài 4 (tự viết lại):** viết `values_divisible(nums, k)` trả về **danh sách các giá trị khác nhau** (đã sắp xếp tăng) có tần suất chia hết cho `k`.

**Bài 5 (mini challenge):** viết `sum_by_predicate(nums, pred)` nhận một hàm `pred(freq) -> bool` và cộng `v * freq` cho nhóm thỏa `pred`. Dùng nó để biểu diễn cả "tần suất chia hết cho k" và "tần suất bằng đúng 1".

In [26]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().sumDivisibleByK([2, 2, 2, 2, 5, 5], 2))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def sum_unique_todo(nums):
    # TODO: dùng Counter, chỉ cộng giá trị có tần suất == 1
    return 0

18


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `2` xuất hiện 4 lần, `5` xuất hiện 2 lần; cả `4` và `2` đều chia hết cho `2`. Bài 2: tần suất `1:2, 2:1, 3:3`.

**Mức 2 (vừa):** Bài 3: điều kiện là `c == 1`. Bài 4: duyệt `Counter(nums).items()` rồi `sorted` kết quả. Bài 5: truyền `pred` vào điều kiện `if pred(c)`.

**Mức 3 (gần đáp án):** Bài 1: `2*4 + 5*2 = 18`. Bài 2: chỉ `3` có tần suất `3` chia hết cho `3`, tổng `9`. Bài 3: `sum_unique([1,2,3,2]) = 4`. Bài 5: `pred = lambda c: c % k == 0`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
from collections import Counter

# Bài 1, 2
assert Solution().sumDivisibleByK([2, 2, 2, 2, 5, 5], 2) == 18
assert Counter([1, 1, 2, 3, 3, 3]) == {1: 2, 2: 1, 3: 3}
assert Solution().sumDivisibleByK([1, 1, 2, 3, 3, 3], 3) == 9

# Bài 3
def sum_unique(nums):
    return sum(v for v, c in Counter(nums).items() if c == 1)
assert sum_unique([1, 2, 3, 2]) == 4 and sum_unique([1, 1, 1, 1, 1]) == 0 and sum_unique([1, 2, 3, 4, 5]) == 15

# Bài 4
def values_divisible(nums, k):
    return sorted(v for v, c in Counter(nums).items() if c % k == 0)
assert values_divisible([1, 2, 2, 3, 3, 3, 3, 4], 2) == [2, 3]
assert values_divisible([1, 2, 3, 4, 5], 2) == []

# Bài 5
def sum_by_predicate(nums, pred):
    return sum(v * c for v, c in Counter(nums).items() if pred(c))
assert sum_by_predicate([4, 4, 4, 1, 2, 3], lambda c: c % 3 == 0) == Solution().sumDivisibleByK([4, 4, 4, 1, 2, 3], 3)
assert sum_by_predicate([1, 2, 3, 2], lambda c: c == 1) == sum_unique([1, 2, 3, 2])
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tổng theo tần suất":**

```python
from collections import Counter
return sum(v * c for v, c in Counter(nums).items() if c % k == 0)
# Đếm tần suất một lần, rồi cộng v * c cho nhóm thỏa điều kiện tần suất.
```

**Checklist trước khi submit:**

- [ ] Điều kiện viết đúng chiều: `c % k == 0` (tần suất chia hết cho `k`)?
- [ ] Cộng `v * c` (mọi lần xuất hiện), không chỉ `v`?
- [ ] Không có nhóm nào đạt -> trả `0` (tổng rỗng)?
- [ ] Không sửa `dict` khi đang duyệt nó?
- [ ] Thử `k = 1`, `k` rất lớn, và mảng một phần tử?
- [ ] Chữ ký đúng `sumDivisibleByK(self, nums: List[int], k: int) -> int`?